# FourBi demo (single notebook)

- Update the paths in the next cell.
- This notebook runs the demo end-to-end in one place.

In [ ]:
from pathlib import Path
import shutil
import torch
from torchvision import transforms

from trainer.fourbi_trainer import FourbiTrainingModule
from data.test_dataset import FolderDataset

# TODO: Update these paths
MODEL_PATH = Path('ckpts/3d22_HDIBCO18.pth')
SRC_PATH = Path('/path/to/input_or_folder')  # file or folder
DST_PATH = Path('output_demo')

PATCH_SIZE = 512
EVAL_BATCH_SIZE = 8
FORCE_CPU = False  # set True to force CPU

In [ ]:
# Resolve device
if FORCE_CPU:
    device = torch.device('cpu')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DST_PATH.mkdir(parents=True, exist_ok=True)

# Ensure source is a folder (copy single image if needed)
if SRC_PATH.is_file():
    tmp_dir = DST_PATH / 'tmp_src'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    tmp_img = tmp_dir / SRC_PATH.name
    if not tmp_img.exists():
        shutil.copy2(SRC_PATH, tmp_img)
    src_folder = tmp_dir
else:
    src_folder = SRC_PATH

fourbi = FourbiTrainingModule(config={'resume': str(MODEL_PATH)}, device=device, make_loaders=False)

dataset = FolderDataset(src_folder, patch_size=PATCH_SIZE, overlap=True, transform=transforms.ToTensor())
fourbi.test_data_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

fourbi.config['test_patch_size'] = PATCH_SIZE
fourbi.config['test_stride'] = PATCH_SIZE // 2
fourbi.config['eval_batch_size'] = EVAL_BATCH_SIZE

In [ ]:
# Run inference and save outputs
for i, sample in enumerate(fourbi.folder_test()):
    key = list(sample.keys())[0]
    img, pred, gt = sample[key]
    src_img_path = Path(key)

    dst_img_path = DST_PATH / (src_img_path.stem + '.png')
    pred.save(str(dst_img_path))
    print(f'({i + 1}/{len(dataset)}) Saving {dst_img_path}')

print('Done.')